In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

K = 100
TOP_M = 10
TEST_MIN_YEAR = 1994
TEST_MAX_YEAR = 2016

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")

def load_all(data_dir: str) -> pd.DataFrame:
    csv_files = sorted(glob.glob(os.path.join(data_dir, "y*.csv")))
    dfs = []
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        if not re.match(r"y\d{4}\.csv", fname):
            continue
        dfs.append(pd.read_csv(fpath))
    if not dfs:
        raise ValueError("No y*.csv files found.")
    return pd.concat(dfs, ignore_index=True)

df_all = load_all(DATA_DIR)

test_mask = (df_all["yy"] >= TEST_MIN_YEAR) & (df_all["yy"] <= TEST_MAX_YEAR)
df_test = df_all.loc[test_mask].copy().reset_index(drop=True)
if df_test.empty:
    raise ValueError("No rows found in the selected test interval.")

X_test = df_test

In [ ]:
import pickle
from SR import compute_single_node_sr

top_n = 10
N_ESTIMATORS = 200
MAX_DEPTH = 4
MAX_FEATURES = "sqrt"

OUT_DIR = os.path.join(
    BASE_DIR,
    f"results_ne{N_ESTIMATORS}_md{MAX_DEPTH}_mf{MAX_FEATURES}",
)

save_path = os.path.join(OUT_DIR, "top_items_by_row_by_k.pkl")
if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        top_items_by_row_by_k = pickle.load(f)
else:
    top_items_by_row_by_k = {}

fig, ax = plt.subplots(figsize=(22, 7))
bg_color = "#EBEBEB"
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)

excess_path = os.path.join(OUT_DIR, f"EXCESS_ret_k{K}.csv")
df_excess = pd.read_csv(excess_path)
numeric_cols = df_excess.select_dtypes(include=["number"]).columns

sr_pairs = [(col, compute_single_node_sr(excess_path, col)) for col in numeric_cols]
if not sr_pairs:
    raise ValueError("No numeric columns found for SR calculation.")

max_row_id, max_sr = max(sr_pairs, key=lambda x: x[1])
try:
    max_row_id_key = int(max_row_id)
except (TypeError, ValueError):
    max_row_id_key = max_row_id

top_pairs = sorted(sr_pairs, key=lambda x: x[1], reverse=True)[:top_n]

top_anchor_ids = pd.to_numeric([i for i, _ in top_pairs], errors="coerce")
top_anchor_ids = [int(i) for i in top_anchor_ids if pd.notna(i)][:top_n]

top_items_for_k = top_items_by_row_by_k.get(K, {})
for anchor_id in top_anchor_ids:
    top_items = top_items_for_k.get(anchor_id, {})
    if not top_items:
        continue

    rows = []
    for month, items in top_items.items():
        for item_id, weight in items:
            rows.append({"month": month, "item_id": item_id, "weight": weight})

    neighbors_df = pd.DataFrame(rows)
    if not neighbors_df.empty:
        neighbors_df["item_id"] = pd.to_numeric(neighbors_df["item_id"], errors="coerce")
        neighbors_df = neighbors_df.dropna(subset=["item_id"]).copy()
        neighbors_df["item_id"] = neighbors_df["item_id"].astype(int)

        max_index = len(X_test) - 1
        valid_mask = (neighbors_df["item_id"] >= 0) & (neighbors_df["item_id"] <= max_index)
        neighbors_df = neighbors_df.loc[valid_mask].copy()

        neighbor_full_rows = X_test.iloc[neighbors_df["item_id"].to_numpy()].reset_index(drop=True)
        neighbors_df = pd.concat([neighbors_df.reset_index(drop=True), neighbor_full_rows], axis=1)

    neighbors_path = os.path.join(OUT_DIR, f"top_items_k{K}_anchor_id{anchor_id}.csv")
    neighbors_df.to_csv(neighbors_path, index=False)

best_idx = int(max_row_id_key)
print(f"k={K} | max SR row_id={max_row_id} | SR={max_sr:.4f}")
print(X_test.iloc[[best_idx]][["permno"]])

sr_pairs_sorted = sorted(top_pairs, key=lambda x: x[1], reverse=True)
sorted_i = [i for i, _ in sr_pairs_sorted]
sorted_sr = [sr for _, sr in sr_pairs_sorted]
rank = list(range(1, len(sorted_sr) + 1))

top50_df = pd.DataFrame({
    "rank": rank,
    "anchor_id": pd.to_numeric(sorted_i, errors="coerce"),
    "sr": sorted_sr,
})
top50_df = top50_df.dropna(subset=["anchor_id"]).copy()
top50_df["anchor_id"] = top50_df["anchor_id"].astype(int)
max_index = len(X_test) - 1
top50_df = top50_df[(top50_df["anchor_id"] >= 0) & (top50_df["anchor_id"] <= max_index)].copy()

full_rows_df = X_test.iloc[top50_df["anchor_id"].to_numpy()].reset_index(drop=True)
result_df = pd.concat(
    [top50_df[["rank", "anchor_id", "sr"]].reset_index(drop=True), full_rows_df],
    axis=1,
)

if "permno" not in result_df.columns:
    raise KeyError("Column 'permno' is missing in result_df. Check X_test.")

file_path = os.path.join(OUT_DIR, f"top50_anchor_fullrows_k{K}.csv")
result_df.to_csv(file_path, index=False)

if "LME" in result_df.columns:
    lme_vals = pd.to_numeric(result_df["LME"], errors="coerce")
    finite_lme = lme_vals.dropna()

    if not finite_lme.empty:
        norm = plt.Normalize(vmin=0.0, vmax=0.5, clip=True)
        bar_colors = [
            plt.cm.Blues(0.25 + 0.70 * norm(v)) if pd.notna(v) else (0.7, 0.7, 0.7, 1.0)
            for v in lme_vals
        ]
        sm = plt.cm.ScalarMappable(cmap="Blues", norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.01)
        cbar.set_label("LME")
        cbar.set_ticks([0, 0.25, 0.5])
    else:
        bar_colors = "#6C757D"

    ax.bar(
        rank,
        sorted_sr,
        color=bar_colors,
        edgecolor="white",
        linewidth=0.5,
        width=0.82,
        label=f"SR (k = {K})",
    )
else:
    ax.bar(
        rank,
        sorted_sr,
        color="#20B2B8",
        edgecolor="white",
        linewidth=0.5,
        width=0.82,
        label=f"SR (k = {K})",
    )

ax.set_title(f"n_estimators = {N_ESTIMATORS}; k = {K}; depth = {MAX_DEPTH}", pad=18)
ax.set_xlabel("Rank (1 = highest SR)")
ax.set_ylabel("Monthly Sharpe Ratio (SR)")
ax.set_xticks(rank)
ax.set_xticklabels([str(r) for r in rank])
ax.set_xlim(0.5, len(rank) + 0.5)
ax.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
def load_factor_matrix(factor_path: str) -> pd.DataFrame:
    factor_file = os.path.join(factor_path, "tradable_factors.csv")
    return pd.read_csv(factor_file).iloc[360:636].reset_index(drop=True)

def get_factors(factor_mat: pd.DataFrame, model: str) -> np.ndarray:
    if model == "FF3":
        return factor_mat.iloc[:, 1:4].to_numpy()
    if model == "FF5":
        return factor_mat.iloc[:, [1, 2, 3, 5, 6]].to_numpy()
    if model == "FF11":
        return factor_mat.iloc[:, 1:12].to_numpy()
    raise ValueError(f"Unknown model: {model}")

def alpha_tstat(y: np.ndarray, factors: np.ndarray) -> tuple[float, float]:
    y = np.asarray(y, dtype=float).reshape(-1)
    factors = np.asarray(factors, dtype=float)
    X = np.column_stack([np.ones(factors.shape[0]), factors])
    n, k = X.shape
    if n <= k:
        raise ValueError(f"Not enough observations: n={n}, k={k}")
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    sigma2 = float(resid.T @ resid) / (n - k)
    se = np.sqrt(np.diag(sigma2 * np.linalg.pinv(X.T @ X)))
    if se[0] == 0:
        raise ValueError("Intercept standard error is zero; t-stat undefined.")
    return float(beta[0]), float(beta[0] / se[0])

def load_top_anchor_meta(top50_path: str, top_m: int) -> pd.DataFrame:
    if not os.path.exists(top50_path):
        raise FileNotFoundError(f"Missing file: {top50_path}")
    top50_df = pd.read_csv(top50_path)
    missing = {"anchor_id", "permno", "sr"} - set(top50_df.columns)
    if missing:
        raise KeyError(f"Missing columns in {top50_path}: {missing}")
    top50_df[["anchor_id", "permno", "sr"]] = top50_df[["anchor_id", "permno", "sr"]].apply(pd.to_numeric, errors="coerce")
    top50_df = top50_df.dropna(subset=["anchor_id", "permno", "sr"]).copy()
    top50_df[["anchor_id", "permno"]] = top50_df[["anchor_id", "permno"]].astype(int)
    return (
        top50_df.sort_values("sr", ascending=False)[["anchor_id", "permno", "sr"]]
        .drop_duplicates(subset=["anchor_id"])
        .head(top_m)
        .reset_index(drop=True)
    )

def top_anchor_summary_df(summary_rows: list[dict], top_anchor_meta: pd.DataFrame, value_col: str) -> pd.DataFrame:
    summary_df = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame(columns=["anchorid", value_col])
    if summary_df.empty:
        return summary_df
    return summary_df.merge(
        top_anchor_meta[["anchor_id", "permno"]].rename(columns={"anchor_id": "anchorid"}),
        on="anchorid",
        how="left",
    )
FACTOR_PATH = "/Users/karin/VSCode/Data_NEU/factor"
FF_MODELS = ["FF3", "FF5", "FF11"]
MODEL_COLORS = {"FF3": "#A7C7E7", "FF5": "#BFD67A", "FF11": "#D8CCF2"}

excess_path = os.path.join(OUT_DIR, f"EXCESS_ret_k{K}.csv")
top50_path = os.path.join(OUT_DIR, f"top50_anchor_fullrows_k{K}.csv")

df_excess = pd.read_csv(excess_path)
top_sr_df = load_top_anchor_meta(top50_path, 50)
factor_mat = load_factor_matrix(FACTOR_PATH)

selected_permnos = top_sr_df["permno"].tolist()
anchor_cols = top_sr_df["anchor_id"].astype(str).tolist()
rank = list(range(1, len(anchor_cols) + 1))

model_to_tstats: dict[str, list[float]] = {}
anchor_model_stats: dict[str, dict[str, float]] = {
    str(int(row.anchor_id)): {"sr": float(row.sr)}
    for _, row in top_sr_df.iterrows()
}
for model in FF_MODELS:
    factors = get_factors(factor_mat, model)
    t_stats = []
    for col in anchor_cols:
        if col not in df_excess.columns:
            raise KeyError(f"Column '{col}' from {top50_path} not found in {excess_path}")
        alpha, t = alpha_tstat(df_excess[col].to_numpy()[: factors.shape[0]], factors)
        t_stats.append(t)
        if col in anchor_model_stats:
            anchor_model_stats[col][f"alpha{model}"] = float(alpha)
            anchor_model_stats[col][f"tstat{model}"] = float(t)
    model_to_tstats[model] = t_stats

alpha_tstat_df_topm = pd.DataFrame([
    {"anchorid": int(aid), **vals}
    for aid, vals in anchor_model_stats.items()
]).sort_values("sr", ascending=False).reset_index(drop=True)

alpha_tstat_display = alpha_tstat_df_topm.merge(
    top_sr_df[["anchor_id", "permno"]].rename(columns={"anchor_id": "anchorid"}),
    on="anchorid",
    how="left",
)


In [ ]:
from weighted_characteristics_monthly import compute_weighted_characteristics_by_month, default_output_path

top50_path = os.path.join(OUT_DIR, f"top50_anchor_fullrows_k{K}.csv")
top_anchor_meta = load_top_anchor_meta(top50_path, TOP_M)
if top_anchor_meta.empty:
    raise ValueError("No valid anchor IDs found for Top-M.")

summary_rows = []
for row in top_anchor_meta.itertuples(index=False):
    anchor_id = int(row.anchor_id)
    input_csv = os.path.join(OUT_DIR, f"top_items_k{K}_anchor_id{anchor_id}.csv")
    if not os.path.exists(input_csv):
        print(f"Skipped (missing file): {input_csv}")
        continue
    try:
        out = compute_weighted_characteristics_by_month(pd.read_csv(input_csv))
    except Exception as e:
        print(f"Skipped (error) for anchor_id={anchor_id}: {e}")
        continue
    output_csv = default_output_path(input_csv)
    out.to_csv(output_csv, index=False)
    print(f"Saved weighted monthly characteristics: {output_csv}")
    avg_lme = float(pd.to_numeric(out["LME"], errors="coerce").mean()) if "LME" in out.columns else float("nan")
    if "LME" not in out.columns:
        print(f"Note: LME missing in aggregated file for anchor_id={anchor_id}.")
    summary_rows.append({"anchorid": anchor_id, "avglme": avg_lme})

lme_summary_df_topm = top_anchor_summary_df(summary_rows, top_anchor_meta, "avglme")
if not lme_summary_df_topm.empty:
    lme_display_df = lme_summary_df_topm[["permno", "avglme"]].sort_values("avglme", ascending=False).reset_index(drop=True)
    print(f"LME Top-{TOP_M} summary with permno (in memory: lme_summary_df_topm)")
    print(lme_display_df)
else:
    print("No LME values produced.")

In [ ]:
def compute_turnover_for_anchor(df_top: pd.DataFrame, returns_df: pd.DataFrame) -> pd.DataFrame:
    df_top = df_top.copy()
    df_top["month_idx"] = df_top["yy"] * 100 + df_top["mm"]
    df_top = df_top.sort_values(["month_idx", "permno"]).reset_index(drop=True)
    returns_df = returns_df.copy()
    returns_df["month_idx"] = returns_df["yy"] * 100 + returns_df["mm"]

    rows = []
    months = sorted(df_top["month_idx"].unique())
    for month_prev, month_curr in zip(months, months[1:]):
        df_prev = df_top[df_top["month_idx"] == month_prev].set_index("permno")
        df_curr = df_top[df_top["month_idx"] == month_curr].set_index("permno")
        if df_curr.empty:
            continue

        w_prev = df_prev["weight"] if not df_prev.empty else pd.Series(dtype=float)
        w_curr = df_curr["weight"]
        r_curr = returns_df.loc[returns_df["month_idx"] == month_curr].set_index("permno")["ret"]
        prev_with_returns = w_prev.index.intersection(r_curr.index)
        R_p_t = float((w_prev.loc[prev_with_returns] * r_curr.loc[prev_with_returns]).sum()) if len(prev_with_returns) else 0.0

        turnover_sum = 0.0
        for perm in w_prev.index.union(w_curr.index):
            w_t = float(w_curr.get(perm, 0.0))
            w_t_minus1 = float(w_prev.get(perm, 0.0))
            r_i_t = float(r_curr.get(perm, 0.0)) if perm in r_curr.index else 0.0
            denom = 1.0 + R_p_t
            w_drift = w_t_minus1 if denom == 0.0 else w_t_minus1 * (1.0 + r_i_t) / denom
            turnover_sum += abs(w_t - w_drift)

        rows.append({
            "month_idx": month_curr,
            "turnover": float(turnover_sum),
            "R_p_t": float(R_p_t),
            "n_assets_prev": int(len(w_prev)),
            "n_assets_curr": int(len(w_curr)),
            "missing_asset_returns": int(max(0, len(w_prev) - len(prev_with_returns))),
        })

    return pd.DataFrame(rows)

top_anchor_ids = top_anchor_meta["anchor_id"].tolist() if not top_anchor_meta.empty else []
print(f"Looking for top-items files for K={K}...")
print(f"Top anchor IDs: {top_anchor_ids}")

required_return_cols = {"yy", "mm", "permno", "ret"}
missing_return_cols = required_return_cols - set(df_test.columns)
if missing_return_cols:
    raise KeyError(f"Missing columns in df_test for returns: {missing_return_cols}")
returns_df = df_test[["yy", "mm", "permno", "ret"]].copy()

turnover_rows = []
for anchor_id in top_anchor_ids:
    top_items_path = os.path.join(OUT_DIR, f"top_items_k{K}_anchor_id{anchor_id}.csv")
    if not os.path.exists(top_items_path):
        print(f"Skipped (missing file): {top_items_path}")
        continue
    try:
        turnover_df = compute_turnover_for_anchor(pd.read_csv(top_items_path), returns_df)
    except Exception as e:
        print(f"Skipped (error) for anchor_id={anchor_id}: {e}")
        continue
    if turnover_df.empty:
        print(f"No turnover rows for anchor_id={anchor_id}.")
        continue
    avg_turnover = float(turnover_df["turnover"].mean())
    print(f"anchor_id={anchor_id}: avg_turnover={avg_turnover:.6f} (n_months={len(turnover_df)})")
    turnover_rows.append({"anchorid": int(anchor_id), "avgturnover": avg_turnover})

turnover_summary_df_topm = top_anchor_summary_df(turnover_rows, top_anchor_meta, "avgturnover")
if not turnover_summary_df_topm.empty:
    turnover_display_df = turnover_summary_df_topm[["permno", "avgturnover"]].sort_values("avgturnover", ascending=False).reset_index(drop=True)
else:
    print("No turnover values produced.")

base_df = top_anchor_meta[["anchor_id", "permno", "sr"]].rename(columns={"anchor_id": "anchorid"})
combined_df = base_df.merge(alpha_tstat_df_topm, on="anchorid", how="left")
combined_df = combined_df.merge(lme_summary_df_topm, on="anchorid", how="left") if not lme_summary_df_topm.empty else combined_df.assign(avglme=np.nan)
combined_df = combined_df.merge(turnover_summary_df_topm, on="anchorid", how="left") if not turnover_summary_df_topm.empty else combined_df.assign(avgturnover=np.nan)

ordered_cols = [
    "anchorid", "permno", "sr",
    "alphaFF3", "tstatFF3", "alphaFF5", "tstatFF5", "alphaFF11", "tstatFF11",
    "avglme", "avgturnover",
]
combined_df = combined_df[[col for col in ordered_cols if col in combined_df.columns]]
combined_csv_path = os.path.join(OUT_DIR, f"top{TOP_M}_anchor_metrics_combined_k{K}.csv")
combined_df.to_csv(combined_csv_path, index=False)